# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
result = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1")
print(result)
month_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
print(month_path)

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature vector built (real warehouse data):** Queried the FlyRank warehouse (`fact_content_daily_performance`, March 2026 partition, 9,841,378 daily rows) via DuckDB over the Hugging Face-hosted parquet files. Built a day-15 split: features aggregated from March 1-15 (impressions, clicks, avg position, CTR, plus GA4 engagement signals filtered on `ga4_data_available` to avoid treating pre-tracking zeros as real zeros), and the decline label defined from March 16-31 clicks (a page is "declining" if its second-half clicks dropped more than 10% versus its first-half clicks). This avoids future leakage by construction, since every feature is computed strictly before the label window begins.

In [12]:
feature_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) as impressions_15d,
        SUM(gsc_clicks) as clicks_15d,
        AVG(gsc_avg_position) as avg_position_15d,
        SUM(CASE WHEN ga4_data_available THEN ga4_sessions ELSE NULL END) as sessions_15d,
        SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions ELSE NULL END) as engaged_sessions_15d,
        SUM(CASE WHEN ga4_data_available THEN scroll_events ELSE NULL END) as scroll_events_15d,
        MAX(ga4_data_available) as has_ga4
    FROM read_parquet('{month_path}')
    WHERE report_date <= DATE '2026-03-15'
      AND gsc_data_available = TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
"""
features_df = con.sql(feature_query).df()

label_query = f"""
    SELECT content_hash_id, SUM(gsc_clicks) as clicks_1631
    FROM read_parquet('{month_path}')
    WHERE report_date >= DATE '2026-03-16' AND gsc_data_available = TRUE
    GROUP BY content_hash_id
"""
labels_df = con.sql(label_query).df()

merged = features_df.merge(labels_df, on='content_hash_id', how='left')
merged['clicks_1631'] = merged['clicks_1631'].fillna(0)
merged['ctr_15d'] = merged['clicks_15d'] / merged['impressions_15d']
merged['declining'] = (merged['clicks_1631'] < merged['clicks_15d'] * 0.9).astype(int)

print(f"Rows: {len(merged)}, has_ga4 rate: {merged['has_ga4'].mean():.3f}")
print(f"Base rate (declining): {merged['declining'].mean():.3f}")
merged.head()

Rows: 151981, has_ga4 rate: 0.442
Base rate (declining): 0.187


,content_hash_id,client_hash_id,impressions_15d,clicks_15d,avg_position_15d,sessions_15d,engaged_sessions_15d,scroll_events_15d,has_ga4,clicks_1631,ctr_15d,declining
0,content_05597932fe4da067,client_73cda7b4e4f265ea,18.0,0.0,4.939394,NaN,NaN,NaN,<NA>,0.0,0.000000,0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,4173.0,6.0,6.327311,NaN,NaN,NaN,<NA>,1.0,0.001438,1
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,89.0,0.0,3.010741,NaN,NaN,NaN,<NA>,0.0,0.000000,0
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,245.0,0.0,3.906852,NaN,NaN,NaN,<NA>,0.0,0.000000,0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,3705.0,3.0,6.473735,NaN,NaN,NaN,<NA>,3.0,0.000810,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Feature availability check:** All four GSC-derived features (impressions_15d, clicks_15d, avg_position_15d, ctr_15d) are 100% available across all 151,981 rows — zero missing. The three GA4-derived features (sessions_15d, engaged_sessions_15d, scroll_events_15d) are missing for 76.1% of rows, which matches exactly with the has_ga4=False rate — confirming the null-handling correctly reflects real tracking gaps rather than silently treating "no GA4 setup" as "zero engagement." Every feature is computed strictly from the report_date ≤ 2026-03-15 window, while the label is computed strictly from report_date ≥ 2026-03-16 — a hard date boundary with zero overlap, confirmed by construction rather than by chance.

In [13]:
print("Feature availability check:\n")

for col in ['impressions_15d', 'clicks_15d', 'avg_position_15d', 'ctr_15d']:
    print(f"{col}: {merged[col].isna().sum()} missing / {len(merged)} total — always available from GSC")

print(f"\nGA4-gated features (only valid where has_ga4=True):")
for col in ['sessions_15d', 'engaged_sessions_15d', 'scroll_events_15d']:
    missing = merged[col].isna().sum()
    print(f"{col}: {missing} missing ({missing/len(merged):.1%}) — matches has_ga4=False rate ({(~merged['has_ga4'].fillna(False)).mean():.1%})")

print(f"\nAvailable-when check: all features computed from report_date <= 2026-03-15,")
print(f"label computed from report_date >= 2026-03-16 — zero overlap, confirmed by construction.")

Feature availability check:

impressions_15d: 0 missing / 151981 total — always available from GSC
clicks_15d: 0 missing / 151981 total — always available from GSC
avg_position_15d: 0 missing / 151981 total — always available from GSC
ctr_15d: 0 missing / 151981 total — always available from GSC

GA4-gated features (only valid where has_ga4=True):
sessions_15d: 115684 missing (76.1%) — matches has_ga4=False rate (76.1%)
engaged_sessions_15d: 115684 missing (76.1%) — matches has_ga4=False rate (76.1%)
scroll_events_15d: 115684 missing (76.1%) — matches has_ga4=False rate (76.1%)

Available-when check: all features computed from report_date <= 2026-03-15,
label computed from report_date >= 2026-03-16 — zero overlap, confirmed by construction.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**The leakage hunt:** Trained an identical Random Forest twice on the same client-grouped split (GroupShuffleSplit by client_hash_id, 70/30). Using only safe, decision-time-available features (impressions_15d, clicks_15d, avg_position_15d, ctr_15d), the model reached AUC = 0.921 — already a strong, honest result. Deliberately adding the actual future click count (clicks_1631 — the exact value used to construct the label) as a feature pushed AUC to 0.998, a 0.077 jump. This is what label leakage looks like in practice: a model that appears near-perfect not because it learned a real pattern, but because it was handed the answer. This deliberately-leaky feature is removed for all subsequent modeling — only the honest 0.921 AUC configuration is used going forward.

In [14]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

safe_features = ['impressions_15d', 'clicks_15d', 'avg_position_15d', 'ctr_15d']
model_df = merged.dropna(subset=safe_features + ['declining']).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]

rf_honest = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf_honest.fit(train_df[safe_features], train_df['declining'])
honest_auc = roc_auc_score(test_df['declining'], rf_honest.predict_proba(test_df[safe_features])[:,1])
print(f"Honest AUC (safe features only): {honest_auc:.3f}")

model_df['LEAKY_future_clicks'] = model_df['clicks_1631']
leaky_features = safe_features + ['LEAKY_future_clicks']
train_df2, test_df2 = model_df.iloc[train_idx], model_df.iloc[test_idx]

rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf_leaky.fit(train_df2[leaky_features], train_df2['declining'])
leaky_auc = roc_auc_score(test_df2['declining'], rf_leaky.predict_proba(test_df2[leaky_features])[:,1])

print(f"Leaky AUC (with future clicks as a feature): {leaky_auc:.3f}")
print(f"\nGap: {leaky_auc - honest_auc:.3f} — this jump is exactly what label leakage looks like.")

Honest AUC (safe features only): 0.921
Leaky AUC (with future clicks as a feature): 0.998

Gap: 0.077 — this jump is exactly what label leakage looks like.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**What I excluded and why:** clicks_1631 (the future click count) is the clearest case — it directly defines the label, and Section 3 proved its inclusion is pure leakage (AUC 0.921 → 0.998). content_hash_id and client_hash_id are pseudonymous IDs, used only for grouped splitting, never as features. Raw report_date is excluded as a feature to avoid the model keying off calendar-specific effects unique to March 2026. GA4-derived features (sessions, engagement, scroll) are excluded from this audit's safe_features set — not because they're unsafe, but because their 76.1% missingness needs a deliberate imputation strategy that belongs in the main modeling notebook rather than this leakage-focused one.

In [15]:
excluded = {
    'clicks_1631 (future clicks)': 'The exact value used to construct the label. Demonstrated above: including it as a feature inflates AUC from 0.921 to 0.998 — pure label leakage.',
    'content_hash_id, client_hash_id': 'Pseudonymous IDs — used only for grouping the train/test split, never as predictive features.',
    'report_date (raw)': 'Excluded as a direct feature since it would let the model key off calendar effects specific to this one month rather than generalizable content signals — used only to define the feature/label window boundary.',
    'sessions_15d, engaged_sessions_15d, scroll_events_15d': 'Excluded from the final safe_features set for this leakage-hunt notebook (kept available but unused) since 76.1% missingness would require imputation choices better suited to the main modeling notebook rather than this audit.',
}
for feat, reason in excluded.items():
    print(f"{feat}: {reason}")

clicks_1631 (future clicks): The exact value used to construct the label. Demonstrated above: including it as a feature inflates AUC from 0.921 to 0.998 — pure label leakage.
content_hash_id, client_hash_id: Pseudonymous IDs — used only for grouping the train/test split, never as predictive features.
report_date (raw): Excluded as a direct feature since it would let the model key off calendar effects specific to this one month rather than generalizable content signals — used only to define the feature/label window boundary.
sessions_15d, engaged_sessions_15d, scroll_events_15d: Excluded from the final safe_features set for this leakage-hunt notebook (kept available but unused) since 76.1% missingness would require imputation choices better suited to the main modeling notebook rather than this audit.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.